<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/Python-2026/Practice/DB_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📚 Вариант 1. SQL + файлы и БД

> **Тема:** библиотека — книги, читатели, выдачи, штрафы. Одна сквозная схема данных от `seed.sql` до финального проекта.
> **Цель:** покрыть **всю теорию** раздела: SQL от `SELECT` до оконных функций и триггеров, работу с файлами (CSV/JSON/JSONL/Excel/Parquet/pickle/XML/YAML/сжатие) и интеграцию с Python (sqlite3, pandas, SQLAlchemy).
> **Схема данных (одна для всех задач):**
> ```sql
> books(id, title, author, year, genre, price, isbn)
> readers(id, name, birth_year, city, email)
> loans(id, book_id, reader_id, loan_date, return_date, days_overdue)
> fines(id, loan_id, amount, paid)
> ```
> **СУБД:** SQLite (встроена, не требует установки). Синтаксис совместим с PostgreSQL/MySQL на 95%.

---

## 🟢 Уровень 0. Setup: создание БД и синтетические данные (0.1–0.5)

**Что изучается:** проектирование схемы (ER, нормализация), `CREATE TABLE`, `PRAGMA foreign_keys`, `CHECK`/`UNIQUE`/`NOT NULL`, `INSERT` вручную, `Faker` для синтетики, `reset.py`, воспроизводимость через `seed`.

**Задача 0.1 (проектирование схемы).** Прежде чем писать SQL, нужно понять, **что** хранить и **как** связать. Нарисуйте ER-диаграмму для библиотеки: сущности `books`, `readers`, `loans`, `fines`; связи «книга — выдача — читатель»; штраф ссылается на выдачу. Для каждого поля обоснуйте тип: почему `year` — `INTEGER`, а не `TEXT`; почему `price` — `REAL`, а не `INTEGER`; почему `id` — `INTEGER PRIMARY KEY`; где нужен `NOT NULL`, где `UNIQUE`, где `CHECK`. Ожидаемый результат: текстовое описание + список полей с типами и ограничениями. В комментариях поясните: что такое 1НФ/2НФ/3НФ на этом примере; почему `author` пока не выносим в отдельную таблицу; когда стоит вынести (если авторов много и они повторяются); что такое «ссылочная целостность». Подсказка: начните с `books`, к ней цепляйте `loans`, к `loans` — `fines`.

**Задача 0.2 (создание БД и таблиц).** Создайте файл `library.db` и в нём — 4 таблицы по схеме из 0.1. Обязательно: `PRAGMA foreign_keys = ON` (в SQLite внешние ключи **выключены по умолчанию**!), `CHECK (year BETWEEN 1500 AND 2100)` на `books.year`, `ON DELETE CASCADE` на `loans.book_id`, `UNIQUE (title, author)` на `books`. Проверьте: `PRAGMA table_info(books)` показывает все поля; попытка вставить `loans.book_id = 9999` (несуществующая книга) даёт `FOREIGN KEY constraint failed`; попытка вставить `books.year = 3000` даёт `CHECK constraint failed`. Ожидаемый результат: 4 таблицы + 2 успешных «падения» на ограничениях. В комментариях поясните: зачем `PRAGMA foreign_keys = ON` (это **не** дефолт в SQLite, в отличие от PostgreSQL); почему `CHECK` лучше проверки в Python (валидация на уровне БД, нельзя обойти); что такое композитный `UNIQUE (title, author)`; что будет с `loans`, если удалить книгу с `CASCADE`. Подсказка: `CREATE TABLE books (id INTEGER PRIMARY KEY, title TEXT NOT NULL, author TEXT NOT NULL, year INTEGER CHECK (year BETWEEN 1500 AND 2100), genre TEXT, price REAL DEFAULT 0, isbn TEXT, UNIQUE (title, author));`.

**Задача 0.3 (ручной seed).** Пустая БД бесполезна — задачи 1–48 не запустятся. Заполните её вручную: 10 книг (Толстой, Достоевский, Булгаков — с годами 1869, 1880, 1967…), 5 читателей (Москва, СПб, Казань), 8 выдач (у некоторых `return_date IS NULL` — книга не возвращена), 3 штрафа (один `paid = 1`, два `paid = 0`). Вставки — пачкой. Проверьте: `SELECT COUNT(*) FROM books` → `10`; `SELECT COUNT(*) FROM loans WHERE return_date IS NULL` → `2`. Ожидаемый результат: все 4 таблицы заполнены. В комментариях поясните: почему пачка быстрее 10 одиночных `INSERT` (одна транзакция); зачем явно перечислять столбцы (иначе порядок = порядок в схеме, ломается при `ALTER`); почему `NULL` в `return_date` — это не `''` и не `0`; как проверить целостность (`PRAGMA foreign_key_check`). Подсказка: `INSERT INTO books (title, author, year, genre, price) VALUES ('Война и мир', 'Л. Толстой', 1869, 'роман', 1200), ('Анна Каренина', 'Л. Толстой', 1877, 'роман', 1100), ...;`.

**Задача 0.4 (Faker: 10 000 строк).** 10 книг хватит, чтобы проверить запрос, но не чтобы увидеть разницу между `O(n)` и `O(log n)` или между `IN` и `EXISTS`. Сгенерируйте синтетику через `Faker` (`pip install faker`): 10 000 книг (`title`, `author`, `year` 1500–2025, `genre` из списка, `price` 100–5000), 2 000 читателей (`name`, `birth_year`, `city`, `email`), 50 000 выдач (случайные `book_id`/`reader_id`, `loan_date` за 2 года, у 70% есть `return_date`, у 20% `days_overdue > 0`). Загрузите через `executemany` (не `to_sql` — он медленный). Обязательно `Faker.seed(42)` и `random.seed(42)` — чтобы результат был воспроизводимым. Проверьте: `SELECT COUNT(*) FROM loans` → `50000`; `SELECT COUNT(*) FROM loans WHERE days_overdue > 0` ≈ `10000`. Ожидаемый результат: полная БД, время генерации < 10 секунд. В комментариях поясните: зачем `seed` (воспроизводимость — тесты не должны «мигать»); почему `executemany` в 10–50 раз быстрее одиночных `INSERT`; зачем генерировать «грязные» данные (пропуски, выбросы, дубликаты) — на чистых данных не научишься `COALESCE`, `IS NULL`, `DISTINCT`; почему `Faker("ru_RU")` даёт русские имена; как не сломать `FOREIGN KEY` (сначала `books` и `readers`, потом `loans`). Подсказка: `from faker import Faker; fake = Faker("ru_RU"); Faker.seed(42); books = [(fake.sentence(nb_words=3), fake.name(), fake.random_int(1500, 2025), fake.random_element(["роман", "детектив", "поэма", "фантастика"]), fake.random_int(100, 5000)) for _ in range(10000)]; cur.executemany("INSERT INTO books (title, author, year, genre, price) VALUES (?, ?, ?, ?, ?)", books)`.

**Задача 0.5 (`reset.py`).** Эксперименты ломают БД — это нормально, но восстанавливать вручную больно. Напишите `reset.py`: (1) удаляет `library.db`, если он есть (`Path.unlink(missing_ok=True)`); (2) выполняет `schema.sql`; (3) выполняет `seed.sql`; (4) печатает `БД готова: 10 книг, 5 читателей, 8 выдач`. Запустите дважды — второй раз тоже должно работать. Ожидаемый результат: `python reset.py` → `БД готова: ...`; повторный запуск → тот же результат. В комментариях поясните: зачем скрипт сброса (быстрый цикл «сломал — починил»); почему `unlink(missing_ok=True)` лучше `if path.exists()`; почему `executescript` выполняет несколько `;`-разделённых запросов; почему в реальных проектах — миграции (`alembic`), а не «дропнуть и пересоздать» (теряются данные); когда «дропнуть и пересоздать» всё-таки ок (dev, тесты, обучение). Подсказка: `from pathlib import Path; Path("library.db").unlink(missing_ok=True); conn = sqlite3.connect("library.db"); conn.executescript(Path("schema.sql").read_text(encoding="utf-8")); conn.executescript(Path("seed.sql").read_text(encoding="utf-8")); conn.commit()`.

---

## 🟢 Уровень 1. SELECT, WHERE, ORDER BY, LIMIT (1–8)

**Что изучается:** `SELECT`, `FROM`, `WHERE`, `ORDER BY`, `LIMIT`, арифметика, сравнения, `AND/OR/NOT`, `AS`, `||`, порядок выполнения запроса.

**Задача 1 (первый SELECT).** Библиотека ведёт таблицу книг, и первое, что нужно — увидеть содержимое. Выведите все столбцы `books`. Ожидаемый вывод: 10 строк со всеми полями. В комментариях поясните: чем `SELECT *` отличается от перечисления полей; почему `SELECT *` — плохая практика (лишние данные, ломается при изменении схемы); что делает `FROM`. Подсказка: `SELECT * FROM books;`.

**Задача 2 (выбор столбцов).** Иногда нужна не вся таблица, а пара столбцов. Выведите `title` и `year`. Ожидаемый вывод: 10 строк с двумя столбцами. В комментариях поясните: почему указание столбцов ускоряет запрос; в каком порядке идут столбцы (в порядке `SELECT`, а не таблицы). Подсказка: `SELECT title, year FROM books;`.

**Задача 3 (WHERE: одно условие).** Читателя интересуют современные книги. Выведите `title`, `author`, `year` для `year >= 2000`. Ожидаемый вывод: 3 строки. В комментариях поясните: почему `WHERE` фильтрует **до** вывода (порядок: `FROM → WHERE → SELECT`); почему алиасы из `SELECT` нельзя использовать в `WHERE`; что такое «предикат». Подсказка: `SELECT title, author, year FROM books WHERE year >= 2000;`.

**Задача 4 (WHERE: арифметика + AND).** Книги с ценой от 500 до 1500 ₽. Выведите `title`, `price`. Ожидаемый вывод: строки в диапазоне. В комментариях поясните: что делает `AND`; почему `BETWEEN` — сахар для `>= AND <=`; что будет, если написать `OR` (условие «расширится»). Подсказка: `SELECT title, price FROM books WHERE price BETWEEN 500 AND 1500;`.

**Задача 5 (арифметика + AS).** Цена книги + 10% надбавки как `price_with_markup`. Выведите `title`, `price`, `price * 1.1 AS price_with_markup`. Ожидаемый вывод: 10 строк с тремя столбцами. В комментариях поясните: что такое `AS` (удобное имя, доступно в `ORDER BY`, но не в `WHERE`); почему арифметика в `SELECT` допустима; что будет без `AS`. Подсказка: `SELECT title, price, price * 1.1 AS price_with_markup FROM books;`.

**Задача 6 (ORDER BY).** Отсортируйте книги по году (сначала новые). Выведите `title`, `year`. Ожидаемый вывод: от `2023` до `1869`. В комментариях поясните: что делает `DESC` и что `ASC` по умолчанию; можно ли сортировать по двум столбцам (`ORDER BY year DESC, title ASC`); где `ORDER BY` в порядке выполнения (после `SELECT`). Подсказка: `SELECT title, year FROM books ORDER BY year DESC;`.

**Задача 7 (LIMIT + ORDER BY).** Топ-3 самых дорогих книг. Выведите `title`, `price`. Ожидаемый вывод: 3 строки. В комментариях поясните: почему `LIMIT` без `ORDER BY` даёт «случайные» строки; что такое «пагинация» (`LIMIT 10 OFFSET 20`); как `LIMIT` работает в PostgreSQL и MySQL (`LIMIT offset, count`). Подсказка: `SELECT title, price FROM books ORDER BY price DESC LIMIT 3;`.

**Задача 8 (конкатенация + алиасы).** Для каталога — строка `«Война и мир» — Л. Толстой`. Используйте `||`. Ожидаемый вывод: 10 строк вида. В комментариях поясните: чем `||` отличается от `+` (в MySQL `+` — арифметика; в PostgreSQL работает `||`, есть `concat()`); что такое `NULL` и как `||` с ним ведёт себя (вернёт `NULL`). Подсказка: `SELECT '«' || title || '» — ' || author AS card FROM books;`.

---

## 🔵 Уровень 2. NULL, строки, DISTINCT, IN/BETWEEN/LIKE, REGEXP (9–16)

**Что изучается:** `NULL` и трёхзначная логика, `IS NULL`/`IS NOT NULL`, `LIKE` с `%`/`_`, `IN`, `BETWEEN`, `DISTINCT`, `UPPER/LOWER/LENGTH/TRIM`, `CASE WHEN`, `REGEXP`.

**Задача 9 (NULL).** В `loans` поле `return_date` может быть `NULL` — книга не возвращена. Выведите `id`, `loan_date`, `return_date` для невозвращённых. Ожидаемый вывод: строки с `return_date IS NULL`. В комментариях поясните: почему `return_date = NULL` возвращает **пусто** (не `TRUE`/`FALSE`); почему `NULL` — не `0` и не `''`; что вернёт `NOT (NULL = 5)`. Подсказка: `SELECT id, loan_date, return_date FROM loans WHERE return_date IS NULL;`.

**Задача 10 (IS NOT NULL + арифметика).** Закрытые выдачи и число дней. Выведите `id`, `loan_date`, `return_date`, `return_date - loan_date AS days_held`. Ожидаемый вывод: строки с `days_held`. В комментариях поясните: почему `return_date - loan_date` работает (в SQLite — юлианский день); как считать разницу в PostgreSQL (`interval`) и MySQL (`DATEDIFF`); почему нельзя вычислить для `NULL`. Подсказка: `SELECT id, loan_date, return_date, return_date - loan_date AS days_held FROM loans WHERE return_date IS NOT NULL;`.

**Задача 11 (LIKE).** Книги Толстого. Выведите `title`, `author` для авторов на «Л.». Ожидаемый вывод: 2 строки. В комментариях поясните: что делает `%`; что делает `_` (ровно один символ); почему `LIKE '%Толстой%'` медленный; как работает `ILIKE` в PostgreSQL. Подсказка: `SELECT title, author FROM books WHERE author LIKE 'Л.%';`.

**Задача 12 (IN + NOT IN).** Книги трёх жанров. Выведите `title`, `genre` для `'роман'`, `'детектив'`, `'поэма'`. Ожидаемый вывод: строки только этих жанров. В комментариях поясните: чем `IN` удобнее цепочки `OR`; **ловушка**: `NOT IN (1, NULL)` возвращает пусто (трёхзначная логика); как этого избежать (`WHERE x IS NOT NULL`). Подсказка: `SELECT title, genre FROM books WHERE genre IN ('роман', 'детектив', 'поэма');`.

**Задача 13 (BETWEEN + даты).** Выдачи за январь 2024. Выведите `id`, `loan_date` для `loan_date` между `'2024-01-01'` и `'2024-01-31'`. Ожидаемый вывод: строки за январь. В комментариях поясните: почему `BETWEEN` **включает** границы; почему для дат лучше `>= '2024-01-01' AND < '2024-02-01'` (если есть время); как хранятся даты (SQLite — `TEXT` ISO, PostgreSQL — `DATE`). Подсказка: `SELECT id, loan_date FROM loans WHERE loan_date BETWEEN '2024-01-01' AND '2024-01-31';`.

**Задача 14 (DISTINCT).** Уникальные жанры. Выведите список без повторов. Ожидаемый вывод: 4 строки. В комментариях поясните: чем `DISTINCT` отличается от `GROUP BY` (`SELECT DISTINCT genre, author` — уникальные **пары**); почему `DISTINCT` дорогой; почему `COUNT(DISTINCT genre)` — отдельная функция. Подсказка: `SELECT DISTINCT genre FROM books;`.

**Задача 15 (строковые функции).** Названия в верхнем регистре и их длина. Выведите `title`, `UPPER(title) AS title_upper`, `LENGTH(title) AS title_len`. Ожидаемый вывод: 3 столбца. В комментариях поясните: какие ещё функции есть (`TRIM`, `SUBSTR`, `REPLACE`, `INSTR`); почему `LENGTH` в SQLite — символы, а в MySQL — байты (в `utf8` русская буква = 2 байта); чем `LENGTH` отличается от `CHAR_LENGTH`. Подсказка: `SELECT title, UPPER(title), LENGTH(title) FROM books;`.

**Задача 16 (CASE WHEN + REGEXP).** Книги делятся на «новые» (с 2010), «средние» (1950–2009), «старые» (до 1950). Выведите `title`, `year`, `CASE WHEN ... AS era`. Дополнительно: найдите книги, где в названии есть цифры, через `REGEXP` (в SQLite — `REGEXP` требует расширения; в PostgreSQL — `~`). Ожидаемый вывод: `era` — одно из трёх значений; список книг с цифрами. В комментариях поясните: что делает `CASE WHEN`; чем `CASE` отличается от `IF` в MySQL; как `CASE` сочетается с агрегатами (`SUM(CASE WHEN ... THEN 1 ELSE 0 END)`); почему `REGEXP` медленный. Подсказка: `CASE WHEN year >= 2010 THEN 'новая' WHEN year >= 1950 THEN 'средняя' ELSE 'старая' END AS era`; в PostgreSQL: `WHERE title ~ '[0-9]'`.

---

## 🟣 Уровень 3. Агрегация: COUNT, SUM, AVG, GROUP BY, HAVING, PIVOT (17–24)

**Что изучается:** агрегатные функции, `GROUP BY`, `HAVING`, «правило полной группы», `COUNT(*)` vs `COUNT(col)`, `COUNT(DISTINCT ...)`, `SUM(CASE WHEN ...)` (pivot).

**Задача 17 (COUNT).** Сколько всего книг. Выведите `COUNT(*) AS total_books`. Ожидаемый вывод: `10`. В комментариях поясните: чем `COUNT(*)` отличается от `COUNT(column)` (второй **не** считает `NULL`); почему `COUNT(1)` = `COUNT(*)`; почему `COUNT` возвращает **одну строку**. Подсказка: `SELECT COUNT(*) AS total_books FROM books;`.

**Задача 18 (SUM, AVG, MIN, MAX).** Вся статистика по ценам. Выведите `SUM(price)`, `AVG(price)`, `MIN(price)`, `MAX(price)`. Ожидаемый вывод: одна строка с 4 числами. В комментариях поясните: что делает `AVG` с `NULL` (игнорирует); почему `SUM` на пустой таблице вернёт `NULL`, а не `0`; как округлить (`ROUND(AVG(price), 2)`). Подсказка: `SELECT SUM(price), AVG(price), MIN(price), MAX(price) FROM books;`.

**Задача 19 (GROUP BY: база).** Число книг по жанрам. Выведите `genre`, `COUNT(*) AS cnt`. Ожидаемый вывод: 4 строки. В комментариях поясните: что делает `GROUP BY`; почему в `SELECT` можно выводить только группирующий столбец + агрегаты; что будет, если вывести `title` без агрегата (SQLite «простит», стандарт — ошибка). Подсказка: `SELECT genre, COUNT(*) AS cnt FROM books GROUP BY genre;`.

**Задача 20 (GROUP BY + ORDER BY).** Жанры по числу книг. Выведите `genre`, `cnt` по убыванию. Ожидаемый вывод: от популярного. В комментариях поясните: почему `ORDER BY cnt DESC` работает (алиас из `SELECT` доступен в `ORDER BY`); почему нельзя `WHERE COUNT(*) > 2`. Подсказка: `SELECT genre, COUNT(*) AS cnt FROM books GROUP BY genre ORDER BY cnt DESC;`.

**Задача 21 (HAVING).** Жанры, где книг **больше двух**. Выведите `genre`, `cnt`. Ожидаемый вывод: 1–2 строки. В комментариях поясните: чем `HAVING` отличается от `WHERE` (`WHERE` — строки **до** группировки, `HAVING` — группы **после**); почему `WHERE COUNT(*) > 2` — ошибка; почему в `HAVING` алиас работает в SQLite/MySQL, но не в PostgreSQL. Подсказка: `SELECT genre, COUNT(*) AS cnt FROM books GROUP BY genre HAVING COUNT(*) > 2;`.

**Задача 22 (COUNT DISTINCT).** Сколько **разных** авторов. Выведите `COUNT(DISTINCT author) AS authors`. Ожидаемый вывод: `5` (если Толстой дважды). В комментариях поясните: почему `COUNT(author)` и `COUNT(DISTINCT author)` разные; почему `COUNT(DISTINCT ...)` дороже; как это связано с `SELECT DISTINCT`. Подсказка: `SELECT COUNT(DISTINCT author) AS authors FROM books;`.

**Задача 23 (PIVOT через CASE).** Сколько книг дороже 1000 ₽ и сколько дешевле, одним запросом. Выведите `SUM(CASE WHEN price > 1000 THEN 1 ELSE 0 END) AS expensive`, `SUM(CASE WHEN price <= 1000 THEN 1 ELSE 0 END) AS cheap`. Ожидаемый вывод: две цифры. В комментариях поясните: почему это «условный счётчик»; почему `COUNT(CASE WHEN price > 1000 THEN 1 END)` тоже работает; где это применяется (сводные таблицы, pivot без `PIVOT`). Подсказка: `SELECT SUM(CASE WHEN price > 1000 THEN 1 ELSE 0 END) AS expensive FROM books;`.

**Задача 24 (GROUP BY по двум полям).** Матрица «жанр × десятилетие». Выведите `genre`, `year / 10 * 10 AS decade`, `COUNT(*) AS cnt`. Ожидаемый вывод: строки по каждой комбинации. В комментариях поясните: почему `GROUP BY genre, decade` группирует по **паре**; как `year / 10 * 10` даёт десятилетие (в SQLite `/` на целых — целочисленное; в PostgreSQL — `FLOOR(year / 10) * 10`); как развернуть в pivot (см. задачу 23). Подсказка: `SELECT genre, year / 10 * 10 AS decade, COUNT(*) AS cnt FROM books GROUP BY genre, decade;`.

---

## 🟠 Уровень 4. JOIN + UNION/INTERSECT/EXCEPT (25–32)

**Что изучается:** `INNER/LEFT/RIGHT/FULL JOIN`, `ON` vs `USING`, анти-join (`LEFT JOIN ... IS NULL`), self-join, `CROSS JOIN`, **потеря строк / дубликаты**, `UNION`/`UNION ALL`/`INTERSECT`/`EXCEPT`.

**Задача 25 (INNER JOIN).** В `loans` хранится `book_id` — подтяните название. Выведите `loans.id`, `books.title`, `loans.loan_date`. Ожидаемый вывод: 8 строк. В комментариях поясните: что делает `INNER JOIN` (только совпавшие пары); почему `ON loans.book_id = books.id` — условие связи; почему `INNER JOIN` теряет выдачи с несуществующим `book_id`. Подсказка: `SELECT l.id, b.title, l.loan_date FROM loans l INNER JOIN books b ON l.book_id = b.id;`.

**Задача 26 (LEFT JOIN).** Все книги + выдачи (если есть). Выведите `books.title`, `loans.loan_date`. Ожидаемый вывод: у книг без выдач `loan_date = NULL`. В комментариях поясните: чем `LEFT JOIN` отличается от `INNER`; что будет с порядком (нужен `ORDER BY`); как считать число выдач **с нулями** (`COUNT(l.id)`, не `COUNT(*)`). Подсказка: `SELECT b.title, l.loan_date FROM books b LEFT JOIN loans l ON l.book_id = b.id;`.

**Задача 27 (анти-join).** Книги, которые **никогда** не выдавались. Выведите `books.title`. Ожидаемый вывод: 2 строки. В комментариях поясните: почему `LEFT JOIN ... WHERE l.id IS NULL` — стандартный анти-join; почему `NOT IN` с подзапросом тоже работает, но `LEFT JOIN` быстрее; почему `NOT EXISTS` — самый чистый (см. 36). Подсказка: `SELECT b.title FROM books b LEFT JOIN loans l ON l.book_id = b.id WHERE l.id IS NULL;`.

**Задача 28 (LEFT JOIN + COUNT).** Число выдач для каждой книги, включая `0`. Выведите `books.title`, `COUNT(l.id) AS cnt`. Ожидаемый вывод: у популярных 3–5, у непопулярных 0. В комментариях поясните: почему `COUNT(*)` даст `1` вместо `0` (`LEFT JOIN` создаёт строку с `NULL`, `*` её считает); почему `COUNT(l.id)` игнорирует `NULL`; как поймать на ревью. Подсказка: `SELECT b.title, COUNT(l.id) AS cnt FROM books b LEFT JOIN loans l ON l.book_id = b.id GROUP BY b.id, b.title;`.

**Задача 29 (UNION / INTERSECT / EXCEPT).** Объедините два списка: книги жанра «роман» и книги после 2000. (а) `UNION` — все без дублей; (б) `UNION ALL` — с дублями; (в) `INTERSECT` — только в обоих; (г) `EXCEPT` — только в первом. Ожидаемый вывод: 4 разных списка. В комментариях поясните: чем `UNION` отличается от `UNION ALL` (последний **быстрее**, не удаляет дубли); почему `INTERSECT` и `EXCEPT` редки в MySQL (только с 8.0.31); как это связано с `JOIN` (часто взаимозаменяемы, но `UNION` работает по **строкам**, а `JOIN` — по **связям**). Подсказка: `SELECT title FROM books WHERE genre = 'роман' UNION SELECT title FROM books WHERE year > 2000;`.

**Задача 30 (многотабличный JOIN).** Читатель, книга, дата — 3 таблицы. Выведите `readers.name`, `books.title`, `loans.loan_date`. Ожидаемый вывод: 8 строк. В комментариях поясните: что `JOIN` цепляются друг за другом; почему порядок `JOIN` влияет на план (оптимизатор обычно справляется); почему 3+ таблиц на миллионах строк — повод задуматься о денормализации. Подсказка: `SELECT r.name, b.title, l.loan_date FROM loans l JOIN readers r ON l.reader_id = r.id JOIN books b ON l.book_id = b.id;`.

**Задача 31 (JOIN + WHERE: ловушка LEFT).** Книги, выданные читателям из Москвы. Выведите `books.title`, `readers.name`. Ожидаемый вывод: строки только для Москвы. В комментариях поясните: почему фильтр по `readers` в `WHERE` **после** `JOIN` — ок для `INNER`; почему для `LEFT JOIN` фильтр по правой таблице в `WHERE` «превращает» его в `INNER` (важная ловушка!); как сохранить `LEFT`-семантику (`AND` в `ON`, не `WHERE`). Подсказка: `SELECT b.title, r.name FROM loans l JOIN books b ON l.book_id = b.id JOIN readers r ON l.reader_id = r.id WHERE r.city = 'Москва';`.

**Задача 32 (self-join).** Пары книг одного жанра (без повторов `A-B` и `B-A`). Выведите `b1.title`, `b2.title`, `b1.genre`. Ожидаемый вывод: пары, где `b1.id < b2.id`. В комментариях поясните: что такое self-join (таблица соединяется сама с собой под разными алиасами); зачем `b1.id < b2.id` (иначе каждая пара дважды + `A-A`); где self-join реально нужен (иерархии, «найти дубликаты», «сотрудник-менеджер»). Подсказка: `SELECT b1.title, b2.title, b1.genre FROM books b1 JOIN books b2 ON b1.genre = b2.genre AND b1.id < b2.id;`.

---

## 🔴 Уровень 5. Подзапросы, EXISTS, IN, CTE, WITH RECURSIVE (33–40)

**Что изучается:** скалярные подзапросы, `IN (SELECT ...)`, `EXISTS`/`NOT EXISTS`, `ANY/ALL`, коррелированные подзапросы, `WITH` (CTE), `WITH RECURSIVE`.

**Задача 33 (скалярный подзапрос).** Книги дороже средней цены. Выведите `title`, `price`. Ожидаемый вывод: 3–4 строки. В комментариях поясните: что такое скалярный подзапрос; почему `price > (SELECT AVG(price) FROM books)` работает, а `price > AVG(price)` — нет; как СУБД выполняет подзапрос (обычно один раз, но не всегда). Подсказка: `SELECT title, price FROM books WHERE price > (SELECT AVG(price) FROM books);`.

**Задача 34 (IN с подзапросом).** Читатели, бравшие хотя бы одну книгу. Выведите `readers.name`. Ожидаемый вывод: 5 строк. В комментариях поясните: что делает `IN (SELECT reader_id FROM loans)`; почему `IN` с подзапросом может быть медленнее `EXISTS`; почему `NOT IN` опасен с `NULL`. Подсказка: `SELECT name FROM readers WHERE id IN (SELECT reader_id FROM loans);`.

**Задача 35 (EXISTS).** Перепишите 34 через `EXISTS`. Ожидаемый вывод: тот же. В комментариях поясните: чем `EXISTS` отличается от `IN` («есть ли хоть одна строка» vs «значение входит в список»); почему `EXISTS` быстрее (останавливается на первой строке); почему `EXISTS` не возвращает `NULL`-проблем. Подсказка: `SELECT name FROM readers r WHERE EXISTS (SELECT 1 FROM loans l WHERE l.reader_id = r.id);`.

**Задача 36 (NOT EXISTS).** Читатели, **никогда** не бравшие книг. Выведите `readers.name`. Ожидаемый вывод: 2 строки. В комментариях поясните: почему `NOT EXISTS` предпочтительнее `NOT IN`; как это соотносится с анти-join (27); когда `LEFT JOIN ... IS NULL` всё-таки лучше (проще читать). Подсказка: `SELECT name FROM readers r WHERE NOT EXISTS (SELECT 1 FROM loans l WHERE l.reader_id = r.id);`.

**Задача 37 (коррелированный подзапрос).** Число выдач для каждой книги через подзапрос. Выведите `books.title`, `(SELECT COUNT(*) FROM loans l WHERE l.book_id = b.id) AS cnt`. Ожидаемый вывод: у непопулярных — `0` (не `NULL`). В комментариях поясните: что значит «коррелированный»; почему `COUNT(*)` в подзапросе даёт `0`; почему коррелированные дорогие (запускаются на каждую строку) и когда `LEFT JOIN + GROUP BY` быстрее. Подсказка: `SELECT b.title, (SELECT COUNT(*) FROM loans l WHERE l.book_id = b.id) AS cnt FROM books b;`.

**Задача 38 (CTE).** Перепишите 37 через `WITH`. Ожидаемый вывод: тот же. В комментариях поясните: что такое CTE (`WITH name AS (...)`) и зачем (декомпозиция, читаемость, повторное использование); почему CTE — не «временная таблица»; когда CTE материализуется (`MATERIALIZED` в PostgreSQL) и почему это важно. Подсказка: `WITH book_loans AS (SELECT book_id, COUNT(*) AS cnt FROM loans GROUP BY book_id) SELECT b.title, COALESCE(bl.cnt, 0) FROM books b LEFT JOIN book_loans bl ON b.id = bl.book_id;`.

**Задача 39 (CTE + HAVING).** Читатели, взявшие больше 2 книг. Выведите `name`, `cnt`. Ожидаемый вывод: 2 строки. В комментариях поясните: почему CTE удобен для «многошаговых»; почему `WHERE` после CTE фильтрует агрегированный результат; когда CTE — «узкое место». Подсказка: `WITH reader_counts AS (SELECT reader_id, COUNT(*) AS cnt FROM loans GROUP BY reader_id) SELECT r.name, rc.cnt FROM reader_counts rc JOIN readers r ON r.id = rc.reader_id WHERE rc.cnt > 2;`.

**Задача 40 (WITH RECURSIVE).** Добавьте таблицу `categories(id, name, parent_id)` — иерархию жанров. Выведите все категории с путём от корня. Ожидаемый вывод: строки вида `Русская литература / Роман`. В комментариях поясните: что такое рекурсивный CTE; зачем «база» и «шаг»; где применяется (деревья категорий, оргструктуры, графы); почему в MySQL 8+/PostgreSQL — `WITH RECURSIVE`. Подсказка: `WITH RECURSIVE tree(id, name, path) AS (SELECT id, name, name FROM categories WHERE parent_id IS NULL UNION ALL SELECT c.id, c.name, t.path || ' / ' || c.name FROM categories c JOIN tree t ON c.parent_id = t.id) SELECT * FROM tree;`.

---

## 🟣 Уровень 6. Оконные функции + JSON в SQL (41–48)

**Что изучается:** `OVER()`, `PARTITION BY`, `ORDER BY` в окне, `ROW_NUMBER`, `RANK`, `DENSE_RANK`, `LAG/LEAD`, `SUM/AVG OVER`, `NTILE`, `FIRST_VALUE`/`LAST_VALUE`, **JSON в SQL** (`json1` в SQLite, `JSONB` в PostgreSQL).

**Задача 41 (ROW_NUMBER).** Пронумеруйте выдачи по дате. Выведите `id`, `loan_date`, `ROW_NUMBER() OVER (ORDER BY loan_date) AS rn`. Ожидаемый вывод: 8 строк с номерами 1–8. В комментариях поясните: что делает `OVER()`; почему `ROW_NUMBER` не схлопывает строки (в отличие от `GROUP BY`); чем отличается от `RANK` (см. 43). Подсказка: `SELECT id, loan_date, ROW_NUMBER() OVER (ORDER BY loan_date) AS rn FROM loans;`.

**Задача 42 (PARTITION BY).** Пронумеруйте выдачи **внутри каждого читателя**. Выведите `reader_id`, `id`, `loan_date`, `ROW_NUMBER() OVER (PARTITION BY reader_id ORDER BY loan_date) AS rn`. Ожидаемый вывод: у каждого свои номера 1, 2, 3… В комментариях поясните: что делает `PARTITION BY` (разбивает на группы, но **не схлопывает**); почему это основа «топ-N в группе»; как без оконных пришлось бы делать (self-join/коррелированный — медленно). Подсказка: `SELECT reader_id, id, loan_date, ROW_NUMBER() OVER (PARTITION BY reader_id ORDER BY loan_date) AS rn FROM loans;`.

**Задача 43 (RANK vs DENSE_RANK vs ROW_NUMBER).** Ранжируйте книги по цене тремя способами. Выведите `title`, `price`, три функции. Ожидаемый вывод: при равных ценах `ROW_NUMBER` — 1, 2; `RANK` — 1, 1, 3; `DENSE_RANK` — 1, 1, 2. В комментариях поясните: в чём разница; когда что нужно («первое место делят двое, следующее — третье»); почему `RANK` без `ORDER BY` — ошибка. Подсказка: `SELECT title, price, RANK() OVER (ORDER BY price DESC) AS r, DENSE_RANK() OVER (ORDER BY price DESC) AS dr, ROW_NUMBER() OVER (ORDER BY price DESC) AS rn FROM books;`.

**Задача 44 (топ-N в группе).** Самая дорогая книга в каждом жанре. Выведите `genre`, `title`, `price`. Ожидаемый вывод: 4 строки. В комментариях поясните: почему `GROUP BY genre` + `MAX(price)` теряет `title`; как `ROW_NUMBER() OVER (PARTITION BY genre ORDER BY price DESC) = 1` решает; почему нужен подзапрос или CTE (окно считается **после** `WHERE`). Подсказка: `WITH ranked AS (SELECT genre, title, price, ROW_NUMBER() OVER (PARTITION BY genre ORDER BY price DESC) AS rn FROM books) SELECT genre, title, price FROM ranked WHERE rn = 1;`.

**Задача 45 (LAG/LEAD).** Для каждой выдачи — дата **предыдущей** у того же читателя. Выведите `reader_id`, `loan_date`, `LAG(loan_date) OVER (PARTITION BY reader_id ORDER BY loan_date) AS prev_date`. Ожидаемый вывод: у первой `prev_date = NULL`. В комментариях поясните: что делает `LAG`/`LEAD`; зачем (расчёт интервалов, «что изменилось»); как заменить `NULL` (`COALESCE`). Подсказка: `SELECT reader_id, loan_date, LAG(loan_date) OVER (PARTITION BY reader_id ORDER BY loan_date) AS prev_date FROM loans;`.

**Задача 46 (SUM OVER: накопительная сумма).** Накопительная сумма выдач по дате. Выведите `loan_date`, `COUNT(*) AS daily`, `SUM(COUNT(*)) OVER (ORDER BY loan_date) AS running_total`. Ожидаемый вывод: `daily` за день, `running_total` накопительно. В комментариях поясните: почему `SUM(...) OVER (ORDER BY ...)` даёт накопительную (окно по умолчанию — «все строки до текущей»); чем отличается от `SUM(...) OVER ()`; как сбросить окно (`ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`). Подсказка: `SELECT loan_date, COUNT(*) AS daily, SUM(COUNT(*)) OVER (ORDER BY loan_date) AS running_total FROM loans GROUP BY loan_date;`.

**Задача 47 (AVG OVER).** Для каждой книги — цена и средняя по жанру. Выведите `title`, `genre`, `price`, `AVG(price) OVER (PARTITION BY genre) AS genre_avg`, `price - genre_avg AS diff`. Ожидаемый вывод: 5 столбцов. В комментариях поясните: почему нельзя в одном `SELECT` использовать `price` и `AVG(price)` **без** окна; почему `AVG(...) OVER (PARTITION BY genre)` одинаков для всех книг жанра; как применить («насколько книга дороже средней»). Подсказка: `SELECT title, genre, price, AVG(price) OVER (PARTITION BY genre) AS genre_avg FROM books;`.

**Задача 48 (JSON в SQL).** В `books` добавьте поле `meta TEXT` с JSON: `{"pages": 500, "language": "ru", "tags": ["классика", "роман"]}`. (а) Извлеките `pages` через `json_extract(meta, '$.pages')` (SQLite) или `meta->>'pages'` (PostgreSQL). (б) Найдите книги, где `pages > 400`. (в) Выведите `title`, `pages`, `language`. Ожидаемый вывод: список книг с извлечёнными полями. В комментариях поясните: зачем JSON в SQL (гибкая схема, полуструктурированные данные); чем `JSONB` в PostgreSQL лучше `JSON` (бинарный, индексируется); как создать индекс на JSON-поле (`CREATE INDEX ... ON books ((meta->>'pages'))`); почему не стоит хранить **всё** в JSON (теряется типизация и约束). Подсказка: SQLite — `SELECT json_extract(meta, '$.pages') FROM books;`; PostgreSQL — `SELECT meta->>'pages' FROM books;`.

---

## 🟤 Уровень 7. DDL/DML, VIEW, транзакции, EXPLAIN (49–56)

**Что изучается:** `CREATE TABLE`, типы, `PRIMARY KEY`/`FOREIGN KEY`/`UNIQUE`/`NOT NULL`/`CHECK`/`DEFAULT`, `CREATE INDEX`, `INSERT/UPDATE/DELETE`, `VIEW`, `BEGIN/COMMIT/ROLLBACK`, `UPSERT`, `EXPLAIN QUERY PLAN`.

**Задача 49 (CREATE TABLE + триггер).** Создайте `books` с нуля: `id INTEGER PRIMARY KEY`, `title TEXT NOT NULL`, `author TEXT NOT NULL`, `year INTEGER CHECK (year BETWEEN 1500 AND 2100)`, `genre TEXT`, `price REAL DEFAULT 0`. Дополнительно создайте **триггер** `trg_books_updated`, который при `UPDATE` пишет в `audit_log(book_id, changed_at)` текущее время. Ожидаемый результат: `PRAGMA table_info(books)` показывает поля; после `UPDATE` в `audit_log` появляется запись. В комментариях поясните: что делает `PRIMARY KEY`; зачем `NOT NULL`; что делает `CHECK`; зачем `DEFAULT`; какие типы в SQLite (`INTEGER`, `TEXT`, `REAL`, `BLOB`, `NUMERIC`); **что такое триггер** (`CREATE TRIGGER ... AFTER UPDATE ON ...`), зачем (аудит, каскады, бизнес-правила), почему триггеры опасны (скрытая логика, сложно отлаживать). Подсказка: `CREATE TRIGGER trg_books_updated AFTER UPDATE ON books BEGIN INSERT INTO audit_log (book_id, changed_at) VALUES (NEW.id, datetime('now')); END;`.

**Задача 50 (FOREIGN KEY + каскады).** Создайте `loans` со ссылкой на `books` и `readers`: `book_id INTEGER NOT NULL REFERENCES books(id) ON DELETE CASCADE`, `reader_id INTEGER NOT NULL REFERENCES readers(id)`. Ожидаемый результат: попытка вставить `loans.book_id = 9999` даёт `FOREIGN KEY constraint failed`. В комментариях поясните: что делает `FOREIGN KEY`; что такое `ON DELETE CASCADE`; почему в SQLite `PRAGMA foreign_keys = ON` нужно включать отдельно; почему каскадное удаление опасно (удалил случайно — потерял историю). Подсказка: `CREATE TABLE loans (..., FOREIGN KEY (book_id) REFERENCES books(id) ON DELETE CASCADE);`.

**Задача 51 (CREATE INDEX + EXPLAIN).** Таблица `loans` вырастет до 50 000 строк, запросы по `reader_id` станут медленными. Создайте индекс `idx_loans_reader` на `loans(reader_id)`. Сравните `EXPLAIN QUERY PLAN SELECT * FROM loans WHERE reader_id = 5` до и после. Ожидаемый результат: до — `SCAN loans`, после — `SEARCH loans USING INDEX idx_loans_reader`. В комментариях поясните: что такое индекс (B-tree, `O(log n)`); почему индексы замедляют `INSERT/UPDATE/DELETE`; какие индексы нужны (по `WHERE`, `JOIN`, `ORDER BY`); почему «индекс на всё» — плохо; **что показывает `EXPLAIN QUERY PLAN`** (план выполнения: `SCAN` vs `SEARCH`, использование индекса, порядок `JOIN`). Подсказка: `CREATE INDEX idx_loans_reader ON loans(reader_id); EXPLAIN QUERY PLAN SELECT * FROM loans WHERE reader_id = 5;`.

**Задача 52 (VIEW).** Создайте **представление** `v_books_with_loans`, которое показывает `title`, `author`, `COUNT(loans.id) AS loan_count`. Используйте его в `SELECT * FROM v_books_with_loans WHERE loan_count > 2`. Ожидаемый результат: 2–3 строки. В комментариях поясните: что такое `VIEW` (именованный запрос, «виртуальная таблица»); зачем (инкапсуляция сложных запросов, повторное использование, безопасность); чем `VIEW` отличается от `CTE` (VIEW хранится в БД, CTE — только в запросе); что такое **материализованное** представление (`MATERIALIZED VIEW` в PostgreSQL, в SQLite нет) и когда оно нужно. Подсказка: `CREATE VIEW v_books_with_loans AS SELECT b.title, b.author, COUNT(l.id) AS loan_count FROM books b LEFT JOIN loans l ON l.book_id = b.id GROUP BY b.id;`.

**Задача 53 (UPDATE + WHERE).** Поднимите цену всех книг жанра «роман» на 10%. Ожидаемый результат: `SELECT price FROM books WHERE genre = 'роман'` показывает увеличенные. В комментариях поясните: почему **без** `WHERE` обновится **вся** таблица (частая катастрофа); как проверить, сколько строк обновилось (`changes()` в sqlite3); почему `UPDATE ... WHERE` лучше делать в транзакции (см. 55). Подсказка: `UPDATE books SET price = price * 1.1 WHERE genre = 'роман';`.

**Задача 54 (DELETE + WHERE).** Удалите все книги до 1900 года. Ожидаемый результат: `COUNT(*)` уменьшился. В комментариях поясните: почему `DELETE FROM books` без `WHERE` удалит всё (и как откатить — только из бэкапа); чем `DELETE` отличается от `TRUNCATE` (в SQLite нет `TRUNCATE`); как `ON DELETE CASCADE` «утащит» связанные строки. Подсказка: `DELETE FROM books WHERE year < 1900;`.

**Задача 55 (транзакции).** Переведите 100 ₽ между двумя штрафами: уменьшите один, увеличьте другой. Сначала — с ошибкой посередине, потом — с `ROLLBACK`. Ожидаемый результат: при ошибке изменения откатываются, при успехе — применяются. В комментариях поясните: что такое ACID; зачем `BEGIN`/`COMMIT`/`ROLLBACK`; почему без транзакции можно получить «половину операции»; как это в Python (`conn.commit()`, `conn.rollback()`). Подсказка: `BEGIN; UPDATE fines SET amount = amount - 100 WHERE id = 1; -- ошибка; ROLLBACK;`.

**Задача 56 (UPSERT).** Вставьте книгу, но если такая уже есть (по `title, author`) — обновите `price`. Ожидаемый результат: первый `INSERT` создаёт, второй — обновляет. В комментариях поясните: что делает `ON CONFLICT (title, author) DO UPDATE SET ...`; зачем `UNIQUE`-индекс; чем отличается от `INSERT OR REPLACE` (последний удаляет старую строку → теряются `id` и связи); как в PostgreSQL (`ON CONFLICT`), MySQL (`ON DUPLICATE KEY UPDATE`). Подсказка: `INSERT INTO books (title, author, year, price) VALUES ('X', 'A', 2000, 500) ON CONFLICT(title, author) DO UPDATE SET price = excluded.price;`.

---

## 🔵 Уровень 8. SQL + Python: sqlite3, pandas, SQLAlchemy (57–64)

**Что изучается:** `sqlite3` (connect/cursor/execute/fetchall/commit), параметры (SQL-инъекции), `executemany`, транзакции из Python, `pd.read_sql`, `df.to_sql`, `SQLAlchemy` (`create_engine`, `text`), JOIN → DataFrame.

**Задача 57 (sqlite3).** Подключитесь к `library.db`, выполните `SELECT * FROM books`, выведите результат. Ожидаемый вывод: список кортежей. В комментариях поясните: что делает `sqlite3.connect("library.db")`; зачем `cursor()`; почему `fetchall()` возвращает кортежи, а не словари; как получить словари (`row_factory = sqlite3.Row`). Подсказка: `import sqlite3; conn = sqlite3.connect("library.db"); cur = conn.cursor(); cur.execute("SELECT * FROM books"); print(cur.fetchall())`.

**Задача 58 (SQL-инъекции).** Найдите книгу по названию. Сначала **неправильно** (f-string → инъекция), потом **правильно** (параметры `?`). Ожидаемый вывод: на `' OR '1'='1` неправильная версия вернёт все строки, правильная — ничего. В комментариях поясните: что такое SQL-инъекция (OWASP Top 10); почему `cur.execute(f"... {name}")` — катастрофа; как работают `?` в sqlite3 и `%s` в psycopg2; почему параметры **не** экранируются вручную. Подсказка: `cur.execute("SELECT * FROM books WHERE title = ?", (name,))` — кортеж, даже для одного параметра.

**Задача 59 (executemany).** Вставьте 1000 книг пачкой. Сравните время с 1000 одиночными `INSERT`. Ожидаемый вывод: `executemany` в 10–50 раз быстрее. В комментариях поясните: почему `executemany` быстрее (одна транзакция, меньше накладных); почему одиночные `INSERT` без `commit` тормозят; как отключить автокоммит (`conn.isolation_level = None` + `BEGIN`). Подсказка: `cur.executemany("INSERT INTO books (title, author, year) VALUES (?, ?, ?)", data)`.

**Задача 60 (транзакции в Python).** Продемонстрируйте: без `commit()` изменения не сохраняются; с `commit()` — сохраняются; с `rollback()` при ошибке — откатываются. Ожидаемый вывод: `COUNT(*)` показывает разные значения в трёх сценариях. В комментариях поясните: почему sqlite3 по умолчанию **не** коммитит; что такое «autocommit» и когда он опасен; как `with conn:` автоматически коммитит/откатывает (context manager!). Подсказка: `try: cur.execute(...); conn.commit() except: conn.rollback()`.

**Задача 61 (pd.read_sql).** Загрузите `books` в DataFrame. Сначала весь `SELECT *`, потом с `WHERE`. Ожидаемый вывод: `df.shape`, `df.head()`. В комментариях поясните: что делает `pd.read_sql("SELECT ...", conn)` (нужен connection, не cursor); почему pandas сам приводит типы; как читать большие таблицы кусками (`chunksize=10000`). Подсказка: `import pandas as pd; df = pd.read_sql("SELECT * FROM books WHERE year >= 2000", conn)`.

**Задача 62 (df.to_sql).** Загрузите DataFrame с 5 книгами в `books_new`. Используйте `if_exists="replace"` (и обсудите `"append"` / `"fail"`). Ожидаемый вывод: `COUNT(*) FROM books_new` → `5`. В комментариях поясните: что делает `to_sql`; чем `"replace"` отличается от `"append"` и `"fail"`; зачем `index=False`; почему `to_sql` медленный на больших DataFrame (`method="multi"` или `chunksize`). Подсказка: `df.to_sql("books_new", conn, if_exists="replace", index=False)`.

**Задача 63 (SQLAlchemy).** Перепишите 57 через SQLAlchemy. Ожидаемый вывод: тот же. В комментариях поясните: зачем `SQLAlchemy` (единый API к SQLite/PostgreSQL/MySQL); что делает `create_engine("sqlite:///library.db")`; зачем `text("SELECT ...")` (SQLAlchemy 2.0 требует обёртку); как параметры передаются (`:name` + `{"name": ...}`). Подсказка: `from sqlalchemy import create_engine, text; engine = create_engine("sqlite:///library.db"); with engine.connect() as conn: result = conn.execute(text("SELECT * FROM books"))`.

**Задача 64 (JOIN → DataFrame).** Загрузите `loans JOIN books JOIN readers` в DataFrame и посчитайте число книг у каждого читателя. Ожидаемый вывод: DataFrame `name` + `cnt`, отсортированный. В комментариях поясните: почему лучше агрегировать **в SQL** (меньше данных в память); когда всё-таки в pandas (сложная логика); как `pd.read_sql` с `GROUP BY` возвращает уже агрегированные данные. Подсказка: `df = pd.read_sql("SELECT r.name, COUNT(*) AS cnt FROM loans l JOIN readers r ON l.reader_id = r.id GROUP BY r.name ORDER BY cnt DESC", conn)`.

---

## 🟠 Уровень 9. Файлы: CSV, JSON, JSONL, Excel, Parquet, pickle, XML, YAML, сжатие (65–72)

**Что изучается:** `csv.reader/writer`, `DictReader/DictWriter`, `json.load/dump`, JSONL, `pd.read_csv/to_csv` (`sep`, `encoding`, `dtype`, `parse_dates`, `usecols`), `pd.read_excel/to_excel`, `pd.read_parquet/to_parquet`, `pickle`, `pathlib`, чтение чанками, **XML/YAML**, **сжатие** (`gzip`, `zip`, `tar`).

**Задача 65 (csv.reader).** Прочитайте `books.csv` (`;`, заголовок). Выведите число строк и первые 3. Ожидаемый вывод: `Книг: 100`, первые 3. В комментариях поясните: зачем `newline=""` в `open` (иначе на Windows — пустые строки); почему `csv.reader` возвращает список строк; как передать `delimiter=";"`. Подсказка: `import csv; with open("books.csv", encoding="utf-8", newline="") as f: reader = csv.reader(f, delimiter=";"); next(reader)`.

**Задача 66 (DictReader/DictWriter).** Прочитайте `books.csv` в список словарей, отфильтруйте `year >= 2000`, запишите в `books_modern.csv` с тем же заголовком. Ожидаемый вывод: новый файл с 3 строками. В комментариях поясните: чем `DictReader` удобнее `reader`; зачем `fieldnames` в `DictWriter`; как `writeheader()` пишет заголовок; почему `encoding="utf-8-sig"` иногда нужен (Excel BOM). Подсказка: `reader = csv.DictReader(f, delimiter=";"); writer = csv.DictWriter(out, fieldnames=reader.fieldnames); writer.writeheader(); writer.writerows(filtered)`.

**Задача 67 (pd.read_csv: параметры).** Загрузите `books.csv` в DataFrame: разделитель `;`, только `title, year, price`, `year` как `int`, `loan_date` как `datetime`. Ожидаемый вывод: `df.dtypes` показывает нужные типы. В комментариях поясните: зачем `sep=";"`; зачем `usecols` (память); зачем `dtype={"year": "Int64"}` (иначе `float` из-за `NaN`); зачем `parse_dates`; почему `encoding` нужно указывать явно на Windows (`cp1251`). Подсказка: `pd.read_csv("books.csv", sep=";", usecols=["title", "year", "price"], dtype={"year": "Int64"}, parse_dates=["loan_date"])`.

**Задача 68 (pd.to_csv).** Сохраните DataFrame в `books_out.csv` **без индекса**, с `;`, в `utf-8`. Перечитайте и проверьте, что данные совпадают. Ожидаемый вывод: `df.equals(df2)` → `True`. В комментариях поясните: зачем `index=False`; почему `encoding="utf-8"` лучше `cp1251`; как записать без заголовка (`header=False`); почему `float_format="%.2f"` полезен для денег. Подсказка: `df.to_csv("books_out.csv", sep=";", index=False, encoding="utf-8")`.

**Задача 69 (JSON).** Прочитайте `books.json` (список объектов), отфильтруйте `year >= 2000`, сохраните в `books_modern.json` с `ensure_ascii=False, indent=2`. Ожидаемый вывод: читаемый JSON с кириллицей. В комментариях поясните: зачем `ensure_ascii=False`; что делает `indent=2`; почему JSON не поддерживает `datetime` и `tuple` (нужен `default=str` или кастомный encoder); чем `json.loads` отличается от `json.load`. Подсказка: `import json; with open("books.json", encoding="utf-8") as f: data = json.load(f); json.dump(filtered, out, ensure_ascii=False, indent=2)`.

**Задача 70 (XML / YAML).** Прочитайте `books.xml` (`<book><title>…</title><year>…</year></book>`) через `xml.etree.ElementTree` и `books.yaml` через `pyyaml`. Загрузите в DataFrame. Ожидаемый вывод: `df.shape` и типы. В комментариях поясните: чем XML отличается от JSON (теги, атрибуты, пространства имён); зачем `ElementTree.findall`/`.find`; почему YAML удобнее для конфигов (человекочитаемый, комментарии); почему `yaml.safe_load`, а не `yaml.load` (**безопасность!** — `load` может выполнить код). Подсказка: `import xml.etree.ElementTree as ET; tree = ET.parse("books.xml"); root = tree.getroot(); [b.find("title").text for b in root.findall("book")]`; `import yaml; data = yaml.safe_load(open("books.yaml"))`.

**Задача 71 (Excel).** Прочитайте `books.xlsx` (несколько листов), возьмите лист `books`, отфильтруйте, запишите в новый `books_filtered.xlsx`. Ожидаемый вывод: файл с одним листом. В комментариях поясните: зачем `openpyxl` (`pip install openpyxl`); как прочитать все листы (`sheet_name=None` → dict); зачем `index=False`; почему Excel медленный для больших данных (лучше Parquet/CSV). Подсказка: `pd.read_excel("books.xlsx", sheet_name="books"); df.to_excel("books_filtered.xlsx", index=False)`.

**Задача 72 (Parquet + pickle + чанки + сжатие).** Сравните форматы: (а) сохраните DataFrame в `books.parquet` и прочитайте; (б) сохраните через `pickle` и прочитайте; (в) прочитайте `big_books.csv` чанками по 1000 строк и посчитайте сумму `price`; (г) сожмите `books.csv` в `books.csv.gz` и прочитайте через `pd.read_csv` (pandas понимает сжатие по расширению). Ожидаемый вывод: Parquet быстрее и меньше; pickle работает, но **небезопасен**; сумма по чанкам = сумме по целому; gzip-файл читается прозрачно. В комментариях поясните: почему Parquet — стандарт для больших данных (колоночное хранение, сжатие, схема); почему `pickle` небезопасен (можно внедрить код); зачем `chunksize`; как pandas определяет сжатие (`compression="infer"`); когда `gzip` vs `bz2` vs `xz` (скорость vs степень сжатия). Подсказка: `df.to_parquet("books.parquet"); for chunk in pd.read_csv("big_books.csv", chunksize=1000): total += chunk["price"].sum(); df.to_csv("books.csv.gz", compression="gzip"); pd.read_csv("books.csv.gz")`.

---

## ⚫ Уровень 10. Мини-проекты: end-to-end пайплайны (73–80)

**Что изучается:** интеграция SQL + файлов + Python + pandas; ETL; миграции; бэкапы; нормализация; отчёты; CLI; тесты для данных.

**Задача 73 (CSV → SQLite: ETL).** Дан `books.csv` (100 строк). Загрузите в SQLite `library.db` в таблицу `books`. Проверьте `COUNT(*) = 100`. Ожидаемый вывод: `Загружено: 100`. В комментариях поясните: что такое ETL; почему `to_sql` с `"append"` может задублировать (нужен `"replace"` или `DELETE`); как валидировать данные до загрузки (`df.isna().sum()`, `df.dtypes`); зачем транзакция вокруг загрузки. Подсказка: `df = pd.read_csv("books.csv", sep=";"); df.to_sql("books", conn, if_exists="replace", index=False)`.

**Задача 74 (SQLite → Excel: отчёт).** Отчёт: по каждому жанру — число книг и средняя цена. Сохраните в `report.xlsx` с двумя листами: `summary` и `books`. Ожидаемый вывод: файл с двумя листами. В комментариях поясните: зачем агрегировать **в SQL**; как `pd.ExcelWriter` пишет несколько листов; почему `avg_price` нужно округлить; как добавить форматирование (`openpyxl`). Подсказка: `with pd.ExcelWriter("report.xlsx") as w: summary.to_excel(w, sheet_name="summary", index=False); books.to_excel(w, sheet_name="books", index=False)`.

**Задача 75 (JSON API → SQLite).** Имитируйте приём из API: `requests.get(...)` → JSON → фильтр → `to_sql`. Используйте локальный `books.json` вместо сети. Ожидаемый вывод: данные из JSON в таблице. В комментариях поясните: что такое API; зачем `response.json()`; почему проверять `response.status_code`; как обрабатывать пагинацию; почему нельзя хардкодить токены (`.env` + `python-dotenv`). Подсказка: `data = json.load(open("books.json")); df = pd.DataFrame(data); df.to_sql(...)`.

**Задача 76 (миграция схемы).** Добавьте в `books` столбец `isbn TEXT` через `ALTER TABLE`. Заполните через `UPDATE`. Ожидаемый вывод: `PRAGMA table_info(books)` показывает `isbn`. В комментариях поясните: что делает `ALTER TABLE ... ADD COLUMN` (в SQLite — только добавление); почему в реальных проектах — миграции (`alembic`, `flyway`); как откатить (в SQLite — пересоздать таблицу); почему `NOT NULL` без `DEFAULT` нельзя добавить в непустую таблицу. Подсказка: `ALTER TABLE books ADD COLUMN isbn TEXT; UPDATE books SET isbn = 'unknown' WHERE isbn IS NULL;`.

**Задача 77 (бэкап).** Сделайте бэкап `library.db` в `library_backup.db`, удалите строку, восстановите, проверьте. Ожидаемый вывод: `COUNT(*)` до = после восстановления. В комментариях поясните: как sqlite делает бэкап (`conn.backup(target)` — атомарно); почему нельзя `cp library.db` во время записи; почему в реальных проектах — `pg_dump`/`mysqldump` + расписание; как восстановить. Подсказка: `src = sqlite3.connect("library.db"); dst = sqlite3.connect("library_backup.db"); src.backup(dst)`.

**Задача 78 (нормализация).** Дана плоская `loans_flat(loan_id, book_title, book_author, reader_name, reader_city, loan_date)`. Разбейте на `books`, `readers`, `loans` (3НФ). Загрузите, проверьте JOIN'ом, что всё восстановилось. Ожидаемый вывод: `COUNT(*) FROM loans` = числу строк в `loans_flat`. В комментариях поясните: что такое 1НФ/2НФ/3НФ; зачем нормализация; когда **денормализация** оправдана (аналитика, read-heavy); как проверить корректность разбиения. Подсказка: `books = df[["book_title", "book_author"]].drop_duplicates(); readers = df[["reader_name", "reader_city"]].drop_duplicates(); ...`.

**Задача 79 (CLI-отчёт).** Напишите `report.py`: `--db library.db --out report.csv --genre роман`, делает `SELECT`, сохраняет CSV. Ожидаемый вывод: `python report.py --db library.db --out report.csv --genre роман` создаёт `report.csv`. В комментариях поясните: зачем `argparse`; как параметризовать SQL (`WHERE genre = ?`, не f-string); почему CLI должен логировать (`logging`); как обрабатывать отсутствие файла БД. Подсказка: `parser = argparse.ArgumentParser(); parser.add_argument("--db", required=True); args = parser.parse_args()`.

**Задача 80 (финальный проект).** Соберите всё: (1) создайте `library.db` со схемой; (2) загрузите `books.csv`, `readers.json`, `loans.csv` (три формата — ETL); (3) напишите 5 SQL-запросов: топ-3 книги по выдачам, топ-3 читателя по книгам, средний срок по жанру, сумма штрафов по читателям, книги ни разу не выданные; (4) сформируйте `report.xlsx` с 5 листами; (5) напишите `pytest`-тесты на 3 запроса; (6) сделайте бэкап `library_backup.db`. Ожидаемый результат: работающий пайплайн + отчёт + тесты. В комментариях опишите: как разбить на модули (`db.py`, `etl.py`, `queries.py`, `report.py`, `main.py`); почему тесты данных важны; как масштабируется на PostgreSQL (заменить `sqlite3` на `psycopg2`/`SQLAlchemy`); какие следующие шаги (Airflow, dbt, Superset). Подсказка: начните с `db.py`, потом `etl.py`, `queries.py`, `report.py`, в конце — `main.py`.

---

## 🟤 Уровень 11. Продвинутые темы (81–88)

**Что изучается:** `EXPLAIN ANALYZE`, профилирование запросов, `JSONB` в PostgreSQL, `FULLTEXT`/`GIN`-индексы, партиционирование, репликация (обзорно), хранимые процедуры, безопасность, миграции глубже.

**Задача 81 (EXPLAIN ANALYZE).** Сравните `EXPLAIN QUERY PLAN` (SQLite) и `EXPLAIN ANALYZE` (PostgreSQL) на запросе `SELECT * FROM loans WHERE reader_id = 5`. Ожидаемый вывод: SQLite — `SEARCH ... USING INDEX`; PostgreSQL — план с `Index Scan`, `actual time`, `rows`. В комментариях поясните: чем `EXPLAIN` отличается от `EXPLAIN ANALYZE` (второй **выполняет** запрос); что такое `Seq Scan` vs `Index Scan`; как читать стоимость (`cost=0.00..8.27`); почему `rows` может сильно отличаться от реального (устаревшая статистика). Подсказка: `EXPLAIN ANALYZE SELECT ...` (PostgreSQL); в SQLite — `EXPLAIN QUERY PLAN`.

**Задача 82 (JSONB в PostgreSQL).** Перепишите задачу 48 под PostgreSQL: `meta JSONB`, `meta->>'pages'`, `meta @> '{"language": "ru"}'`, индекс `GIN` на `meta`. Ожидаемый вывод: те же книги + быстрый поиск по JSON. В комментариях поясните: чем `JSONB` лучше `JSON` (бинарный, индексируется, поддерживает операторы `@>`, `?`, `?|`); как создать `GIN`-индекс; когда JSON-поле — оправдано (полуструктурированные данные, меняющаяся схема); когда — нет (теряется типизация). Подсказка: `CREATE INDEX idx_books_meta ON books USING GIN (meta); SELECT * FROM books WHERE meta @> '{"language": "ru"}';`.

**Задача 83 (FULLTEXT / GIN).** Реализуйте полнотекстовый поиск по названиям книг. SQLite — `FTS5` (`CREATE VIRTUAL TABLE books_fts USING fts5(title)`); PostgreSQL — `tsvector` + `GIN`. Ожидаемый вывод: поиск по слову «война» находит «Война и мир». В комментариях поясните: чем `LIKE '%война%'` хуже `FTS` (не использует индекс, не учитывает морфологию); что такое «токенизация», «стемминг»; как `FTS5`/`tsvector` работают под капотом (инвертированный индекс). Подсказка: SQLite — `CREATE VIRTUAL TABLE books_fts USING fts5(title, content='books', content_rowid='id'); INSERT INTO books_fts(books_fts) VALUES('rebuild'); SELECT * FROM books_fts WHERE books_fts MATCH 'война';`.

**Задача 84 (партиционирование).** Таблица `loans` вырастет до 10 млн строк. Разбейте её на партиции по годам (`loans_2023`, `loans_2024`, …). SQLite — не поддерживает партиционирование; PostgreSQL — `PARTITION BY RANGE (loan_date)`. Ожидаемый вывод: запросы по году идут только в нужную партицию. В комментариях поясните: зачем партиционирование (быстрее, легче удалять старые данные); чем `PARTITION BY` отличается от шардирования (вертикальное vs горизонтальное); как проверить, что запрос использует нужную партицию (`EXPLAIN`); почему в SQLite — только вручную через `UNION ALL` или отдельные таблицы. Подсказка: `CREATE TABLE loans (...) PARTITION BY RANGE (loan_date); CREATE TABLE loans_2024 PARTITION OF loans FOR VALUES FROM ('2024-01-01') TO ('2025-01-01');`.

**Задача 85 (хранимые процедуры).** Создайте процедуру `register_loan(book_id, reader_id, loan_date)`, которая вставляет выдачу и проверяет, что книга не выдана. SQLite — `CREATE TRIGGER`; PostgreSQL — `CREATE FUNCTION ... LANGUAGE plpgsql`. Ожидаемый вывод: вызов процедуры создаёт выдачу; повторная выдача той же книги — ошибка. В комментариях поясните: зачем хранимые процедуры (бизнес-логика на уровне БД, меньше трафика); почему в SQLite нет процедур (только триггеры); почему в современных проектах логику чаще держат в приложении (переносимость, тестируемость). Подсказка: PostgreSQL — `CREATE FUNCTION register_loan(...) RETURNS void AS $$ BEGIN INSERT INTO loans ...; END; $$ LANGUAGE plpgsql;`.

**Задача 86 (безопасность).** Разберите 5 уязвимостей: SQL-инъекции (58), слабые пароли, отсутствие `PRAGMA foreign_keys`, открытый порт SQLite-файла, `pickle` из ненадёжного источника. Для каждой — как защититься. Ожидаемый вывод: чек-лист «уязвимость → защита». В комментариях поясните: почему параметры **не** экранируются вручную; зачем `secrets` вместо `random` для токенов; почему `pickle` — это «выполнить произвольный код»; почему SQLite-файл нельзя отдавать по HTTP; зачем `chmod 600` на `.db`. Подсказка: `import secrets; token = secrets.token_hex(32)`.

**Задача 87 (миграции глубже).** Возьмите `alembic` (или `yoyo-migrations` для SQLite). Напишите 3 миграции: (1) создать `books`; (2) добавить `isbn`; (3) создать индекс `idx_books_year`. Прогоните `upgrade head`, `downgrade -1`, `upgrade head`. Ожидаемый вывод: схема меняется, откатывается. В комментариях поясните: зачем миграции (версионирование схемы, командная работа); чем `alembic` лучше ручного `ALTER TABLE`; как откатывать; почему миграции должны быть **идемпотентны** (повторный запуск не ломает). Подсказка: `alembic init migrations; alembic revision -m "add isbn"; alembic upgrade head`.

**Задача 88 (финальный продвинутый проект).** Соберите всё: (1) схема с `VIEW`, `TRIGGER`, `FTS`; (2) миграции через `alembic`; (3) ETL из CSV/JSON/XML; (4) 3 сложных запроса с оконными функциями и CTE; (5) отчёт в Excel + JSON; (6) CLI; (7) `pytest`-тесты; (8) бэкап; (9) `EXPLAIN ANALYZE` до/после индексов. Ожидаемый результат: production-ready пайплайн. В комментариях опишите: как разбить на модули; почему тесты данных важны; как масштабируется на PostgreSQL; какие следующие шаги (Airflow, dbt, Superset, Kafka). Подсказка: начните с миграций, потом ETL, потом запросы, потом отчёты.

---
